# CO₂ emissions — interactive EDA

This notebook loads a **pre-cleaned** slice of the CO₂ dataset, inspects structure and missing values, applies **linear interpolation** along the year axis, exports `Co2_Emission_with_code.xlsx`, and runs the **`CountryProfile`** helper for a single-country dashboard (summary tables + charts).

**Paths:** code uses `../data/...` — keep the kernel working directory as **`notebook/`** (default in VS Code / Cursor when you open this file).

**Tip:** use **+ Markdown** for more sections; **Shift+Enter** renders a cell.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from pathlib import Path
import sys

## Load data

Read `Co2_Emission_pre_drop.xlsx` from `data/processed/`. The `../` prefix is correct when the kernel’s current working directory is **`notebook/`**.

In [2]:
df_raw= pd.read_excel("../data/processed/Co2_Emission_pre_drop.xlsx")
df_raw

,Country,country_code,Region,Indicator Name,1990,1991,1992,1993,1994,1995,...,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
0,Afghanistan,AFG,South Asia,CO2 emissions (metric tons per capita),1.917451e+08,1.676816e+08,9.595774e+07,8.472111e+06,7.554583e+07,6.846796e+07,...,243614036,29650624,259295334,18562366,146235617,172896741,149789334,131694556,1632953,159824372
1,Angola,AGO,Sub-Saharan Africa,CO2 emissions (metric tons per capita),5.536620e+08,5.445386e+08,5.435572e+08,7.089842e+07,8.368044e+08,9.121415e+08,...,976184198,985522312,950695879,1036293852,1099779111,113504405,1031811348,81330073,777674934,792137069
2,Albania,ALB,Europe & Central Asia,CO2 emissions (metric tons per capita),1.819542e+09,1.242810e+08,6.836998e+08,6.383070e+08,6.453552e+08,6.054363e+08,...,1527623663,166942319,150324046,1533630039,1668337371,160377515,1557664358,1788786074,1782738948,169224832
3,Andorra,AND,Europe & Central Asia,CO2 emissions (metric tons per capita),7.521832e+09,7.235379e+08,6.963079e+07,6.724178e+09,6.541579e+09,6.733479e+09,...,6157197775,5850886105,5944654173,5942800412,5807127723,6026181822,6080600282,6104133912,6362975399,6481217432
4,United Arab Emirates,ARE,Middle East & North Africa,CO2 emissions (metric tons per capita),3.019519e+09,3.177850e+09,2.908093e+09,2.927568e+09,3.084933e+09,3.112502e+09,...,1903976975,1850945738,1920780112,2005564757,2005169797,2107764197,2148066861,2076902233,1839067806,1932956328
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186,Samoa,WSM,East Asia & Pacific,CO2 emissions (metric tons per capita),5.528357e+07,6.097561e+08,6.042661e+08,6.582215e+08,5.928069e+08,7.056748e+08,...,1021813006,1066843067,1057703002,1048701723,1092498145,1240245954,1542099372,1586830344,1478626161,1522124134
187,"Yemen, Rep.",YEM,Middle East & North Africa,CO2 emissions (metric tons per capita),5.670374e+07,6.909374e+08,7.047931e+08,6.271049e+08,6.532557e+08,7.060805e+08,...,1098257856,963978488,858490955,1106687715,1062211282,511361652,399363829,359621635,341068402,380633361
188,South Africa,ZAF,Sub-Saharan Africa,CO2 emissions (metric tons per capita),6.729799e+09,6.424622e+09,6.175430e+09,6.219194e+09,6.215847e+09,6.378790e+09,...,8304084027,7869815906,8077957969,8138264312,8212241156,7669937662,7563739495,7641675086,7515678605,7507736092
189,Zambia,ZMB,Sub-Saharan Africa,CO2 emissions (metric tons per capita),3.409296e+08,3.492322e+08,3.372244e+07,2.899561e+08,2.412696e+08,2.341532e+08,...,195502192,217496732,27860069,284057568,304549552,312354964,325114844,404067778,445489133,380717051


## Column cleanup and missing-value audit

Drop the redundant **`Indicator Name`** column, treat **year columns** as all numeric columns after the metadata fields, then list countries that still have **at least one NaN** in the year grid (before imputation).

In [3]:
df = df_raw.drop(columns=["Indicator Name"])
df

,Country,country_code,Region,1990,1991,1992,1993,1994,1995,1996,...,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
0,Afghanistan,AFG,South Asia,1.917451e+08,1.676816e+08,9.595774e+07,8.472111e+06,7.554583e+07,6.846796e+07,6.258803e+07,...,243614036,29650624,259295334,18562366,146235617,172896741,149789334,131694556,1632953,159824372
1,Angola,AGO,Sub-Saharan Africa,5.536620e+08,5.445386e+08,5.435572e+08,7.089842e+07,8.368044e+08,9.121415e+08,1.072168e+09,...,976184198,985522312,950695879,1036293852,1099779111,113504405,1031811348,81330073,777674934,792137069
2,Albania,ALB,Europe & Central Asia,1.819542e+09,1.242810e+08,6.836998e+08,6.383070e+08,6.453552e+08,6.054363e+08,6.123674e+08,...,1527623663,166942319,150324046,1533630039,1668337371,160377515,1557664358,1788786074,1782738948,169224832
3,Andorra,AND,Europe & Central Asia,7.521832e+09,7.235379e+08,6.963079e+07,6.724178e+09,6.541579e+09,6.733479e+09,6.991595e+08,...,6157197775,5850886105,5944654173,5942800412,5807127723,6026181822,6080600282,6104133912,6362975399,6481217432
4,United Arab Emirates,ARE,Middle East & North Africa,3.019519e+09,3.177850e+09,2.908093e+09,2.927568e+09,3.084933e+09,3.112502e+09,3.092803e+09,...,1903976975,1850945738,1920780112,2005564757,2005169797,2107764197,2148066861,2076902233,1839067806,1932956328
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186,Samoa,WSM,East Asia & Pacific,5.528357e+07,6.097561e+08,6.042661e+08,6.582215e+08,5.928069e+08,7.056748e+08,7.595011e+08,...,1021813006,1066843067,1057703002,1048701723,1092498145,1240245954,1542099372,1586830344,1478626161,1522124134
187,"Yemen, Rep.",YEM,Middle East & North Africa,5.670374e+07,6.909374e+08,7.047931e+08,6.271049e+08,6.532557e+08,7.060805e+08,6.981582e+08,...,1098257856,963978488,858490955,1106687715,1062211282,511361652,399363829,359621635,341068402,380633361
188,South Africa,ZAF,Sub-Saharan Africa,6.729799e+09,6.424622e+09,6.175430e+09,6.219194e+09,6.215847e+09,6.378790e+09,6.489192e+09,...,8304084027,7869815906,8077957969,8138264312,8212241156,7669937662,7563739495,7641675086,7515678605,7507736092
189,Zambia,ZMB,Sub-Saharan Africa,3.409296e+08,3.492322e+08,3.372244e+07,2.899561e+08,2.412696e+08,2.341532e+08,1.884421e+08,...,195502192,217496732,27860069,284057568,304549552,312354964,325114844,404067778,445489133,380717051


In [4]:
emission_cols = df.columns[3:]
emission_cols

Index([1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001,
       2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013,
       2014, 2015, 2016, 2017, 2018, 2019],
      dtype='object')

In [5]:
rows_with_nan = df[df[emission_cols].isna().any(axis=1)]
rows_with_nan[["Country"] + list(emission_cols)]

,Country,1990,1991,1992,1993,1994,1995,1996,1997,1998,...,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
52,Eritrea,NaN,NaN,1.948978e+07,2.814220e+08,3.290607e+08,3.629390e+08,3.960906e+08,3.780990e+08,2.764636e+08,...,157706966,177350806,190763128,195035548,196288983,194446714,195465331,210964662,246177237,251635846
59,"Micronesia, Fed. Sts.",NaN,NaN,1.084962e+09,1.058700e+09,1.037442e+09,1.115916e+09,1.107921e+08,1.106266e+09,1.201412e+09,...,1068832829,1160003067,124394767,1321727314,1303004361,1377587623,1542439793,1614914698,150923297,1581569507
94,Kuwait,1.390221e+09,3.849757e+09,NaN,NaN,NaN,2.257291e+09,2.190111e+09,2.180959e+09,2.300776e+08,...,2697965604,2629058751,2547440065,2477326637,231838022,2320895692,2313702198,2236876769,2208921039,220224164
111,Marshall Islands,NaN,NaN,1.624629e+09,1.806213e+09,1.792329e+09,1.783803e+09,1.781367e+09,1.783838e+09,1.985861e+09,...,2483987165,2476824015,2468613355,2459030801,2623157336,2611238875,3118341166,3100615079,3081558706,3061693238
113,Mali,1.183444e+06,1.158007e+06,1.129901e+06,1.100452e+06,1.071250e+05,1.043225e+06,0.000000e+00,9.906520e+05,NaN,...,18472556,201100982,194624453,19574642,199595937,209303734,259943425,279812002,286721356,29657102
123,Namibia,NaN,7.586025e+08,8.042448e+08,9.066953e+08,1.030263e+09,1.087313e+09,1.154278e+09,1.160169e+09,1.166528e+09,...,1477197645,1544026991,1608363844,1696883716,1737465854,1814332366,1751451675,1777224301,1727729453,1691705428
137,Palau,NaN,NaN,1.261114e+09,1.228124e+09,1.195100e+08,1.165637e+09,1.136428e+09,1.110124e+09,1.086425e+09,...,116965575,1183231877,1247519131,1249786961,1248226944,1188791358,1410994469,1459935934,1395790296,1388811733
170,Timor-Leste,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,219475321,224587074,264783303,398859616,451320002,434675741,492090082,490645572,481082052,479460533


## Imputation and conditional export

**Interpolation:** linear fill along each row (`axis=1`) with `limit` and `limit_area` to control how far gaps are filled; remaining incomplete rows are shown explicitly.

**Export:** the following cell writes **`Co2_Emission_with_code.xlsx`** only if that file **does not exist yet**, avoiding silent overwrites.

In [6]:
df[emission_cols]= df[emission_cols].interpolate(
    method="linear",
    axis=1,
    limit=3,
    limit_area="inside"
)
mask = df.isna().any(axis=1)
df[mask]

,Country,country_code,Region,1990,1991,1992,1993,1994,1995,1996,...,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
52,Eritrea,ERI,Sub-Saharan Africa,NaN,NaN,1.948978e+07,2.814220e+08,3.290607e+08,3.629390e+08,3.960906e+08,...,1.577070e+08,1.773508e+08,1.907631e+08,1.950355e+08,1.962890e+08,1.944467e+08,1.954653e+08,2.109647e+08,2.461772e+08,2.516358e+08
59,"Micronesia, Fed. Sts.",FSM,East Asia & Pacific,NaN,NaN,1.084962e+09,1.058700e+09,1.037442e+09,1.115916e+09,1.107921e+08,...,1.068833e+09,1.160003e+09,1.243948e+08,1.321727e+09,1.303004e+09,1.377588e+09,1.542440e+09,1.614915e+09,1.509233e+08,1.581570e+09
111,Marshall Islands,MHL,East Asia & Pacific,NaN,NaN,1.624629e+09,1.806213e+09,1.792329e+09,1.783803e+09,1.781367e+09,...,2.483987e+09,2.476824e+09,2.468613e+09,2.459031e+09,2.623157e+09,2.611239e+09,3.118341e+09,3.100615e+09,3.081559e+09,3.061693e+09
123,Namibia,NAM,Sub-Saharan Africa,NaN,758602519.0,8.042448e+08,9.066953e+08,1.030263e+09,1.087313e+09,1.154278e+09,...,1.477198e+09,1.544027e+09,1.608364e+09,1.696884e+09,1.737466e+09,1.814332e+09,1.751452e+09,1.777224e+09,1.727729e+09,1.691705e+09
137,Palau,PLW,East Asia & Pacific,NaN,NaN,1.261114e+09,1.228124e+09,1.195100e+08,1.165637e+09,1.136428e+09,...,1.169656e+08,1.183232e+09,1.247519e+09,1.249787e+09,1.248227e+09,1.188791e+09,1.410994e+09,1.459936e+09,1.395790e+09,1.388812e+09
170,Timor-Leste,TLS,East Asia & Pacific,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.194753e+08,2.245871e+08,2.647833e+08,3.988596e+08,4.513200e+08,4.346757e+08,4.920901e+08,4.906456e+08,4.810821e+08,4.794605e+08


In [7]:
if not os.path.exists("../data/processed/Co2_Emission_with_code.xlsx"):
    df.to_excel("../data/processed/Co2_Emission_with_code.xlsx", index=False)

## Importing `CountryProfile`

`country_profile.py` lives under **`src/`** and is not installed as a package. The Jupyter kernel resolves imports from the **current working directory**, so the code below finds the repo root (`repo`), prepends **`repo / "src"`** to `sys.path`, and then runs `from country_profile import CountryProfile, country_reference_table`.

This works whether the kernel `cwd` is the **`notebook/`** folder or the **repository root** (it checks for `src/country_profile.py` in the current folder first).

After importing, run the **Help** cells to list all countries and codes.\n\n**Prerequisite:** run the cells above so `Co2_Emission_with_code.xlsx` exists under `data/processed/` (or adjust the export step). The module reads that file using `Path(__file__)` inside `src/country_profile.py`, so the Excel path does not depend on your notebook `cwd`.


In [ ]:
here = Path.cwd()
repo = here if (here / "src" / "country_profile.py").exists() else here.parent

src_dir = repo / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from country_profile import CountryProfile, country_reference_table


## Help — country names and codes

Use this **reference table** to pick the **ISO Alpha-3** code (column `country_code`) for `CountryProfile("XXX")`. Names and regions match the processed file `Co2_Emission_with_code.xlsx`.

Run the cell below after importing; scroll or use the notebook search on the rendered table.


In [ ]:
country_reference_table()


## Example: one-country dashboard

The next cell builds a **`CountryProfile`** for an **ISO 3166-1 alpha-3** code (three letters, e.g. `BRA`, `PRT`, `ERI`).

- **`show_analysis()`** prints summary tables (mean/median/missing years; world and **within-region** rank as `position/total`; share of world as a `%` string) and shows two plots: the country alone, then the same country **highlighted** against other countries in its **region**.

You can also call **`emission_summary()`**, **`rank_in_world_by_total()`**, **`pct_of_world_total()`**, **`plot_emissions_over_time()`**, or **`plot_region_peers_over_time()`** separately. See **`src/country_profile.py`** for details.


In [ ]:
CountryProfile("ERI").show_analysis()
